### Analyse von Earnings Calls: Hybrides Modell aus Machine Learning und Large Language Models zur Verringerung von Distraction Bias
**Bachelorarbeit an der Universität St.Gallen (HSG)**\
[© 2026 Ruben Cardell](https://polyformproject.org/licenses/noncommercial/1.0.0)

---

# Postprocessing – Kontrollversuch

Dieses Notebook wertet den Kontrollversuch zum Distraction Bias aus:

1. Roh-Ausgaben des nicht-anonymisierten LLM-Laufs inspizieren
2. JSON-Objekte in eine Spalte je Sentiment-Dimension auftrennen
3. Sensitivität gegenüber der Anonymisierung testen (Hypothese 1)

In [1]:
# Google Drive einbinden (Ausführung in Google Colab)
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


## 1 Output-Inspektion

In [2]:
# Roh-Ausgaben des nicht-anonymisierten Laufs laden
df = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/output_control.csv')

In [3]:
# Erste Zeilen zur Sichtprüfung
df.head(10)

,episode_id,output
0,CINF_2019_Q1,"```json\n{""relevant"": 0.95, ""clear"": 0.9, ""opt..."
1,SMCI_2020_Q1,"```json\n{""relevant"": 0.9, ""clear"": 0.8, ""opti..."
2,CI_2017_Q2,"```json\n{""relevant"": 0.95, ""clear"": 0.9, ""opt..."
3,BLDR_2021_Q1,"```json\n{""relevant"": 0.95, ""clear"": 0.9, ""opt..."
4,PSX_2018_Q2,"```json\n{""relevant"": 0.95, ""clear"": 0.9, ""opt..."
5,T_2011_Q2,"{""relevant"": 0.9, ""clear"": 0.9, ""optimistic"": ..."
6,PAYX_2024_Q1,"```json\n{""relevant"": 0.9, ""clear"": 0.9, ""opti..."
7,CMS_2012_Q4,"```json\n{""relevant"": 0.95, ""clear"": 0.9, ""opt..."
8,URBN_2013_Q3,"```json\n{""relevant"": 0.95, ""clear"": 0.9, ""opt..."
9,COF_2008_Q1,"{""relevant"": 0.9, ""clear"": 0.8, ""optimistic"": ..."


In [4]:
# Anzahl Zeilen und fehlende Werte prüfen
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   episode_id  1358 non-null   object
 1   output      1358 non-null   object
dtypes: object(2)
memory usage: 21.3+ KB


In [5]:
# Format einer einzelnen Modellausgabe prüfen
df.iloc[0]["output"]

'```json\n{"relevant": 0.95, "clear": 0.9, "optimistic": 0.7, "specific": 0.9, "cautious": 0.8, "assertive": 0.8, "valence": 0.6, "arousal": 0.4, "dominance": 0.8, "price_1d": 0.02, "price_30d": 0.04, "price_90d": 0.06}\n```'

In [6]:
# Zweite Ausgabe zum Vergleich
df.iloc[1]["output"]

'```json\n{"relevant": 0.9, "clear": 0.8, "optimistic": 0.6, "specific": 0.5, "cautious": 0.7, "assertive": 0.6, "valence": 0.3, "arousal": 0.3, "dominance": 0.6, "price_1d": 0.02, "price_30d": 0.03, "price_90d": 0.04}\n```'

## 2 JSON zu Spalten konvertieren

In [7]:
import json
import re

In [8]:
def parse_output(s):
    """Wandelt die Roh-Ausgabe des LLM in ein Dictionary um.

    Das Modell gibt das JSON-Objekt teilweise in einem Markdown-Codeblock aus;
    dessen Auszeichnung wird zuerst entfernt. Nicht parsbare Ausgaben werden
    protokolliert und als None zurückgegeben, sodass sie später als fehlende
    Werte erscheinen.
    """
    s = re.sub(r'^```json\s*|\s*```$', '', s.strip())
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        print(f"Failed to parse: {s}")
        return None

In [9]:
# JSON-Objekte parsen und in eine Spalte je Dimension überführen
parsed = df['output'].apply(parse_output)
features = pd.json_normalize(parsed)  # NaN-Zeilen, wo das Parsen fehlgeschlagen ist

df = pd.concat([df.drop(columns='output'), features], axis=1)

In [10]:
# Erste Zeilen der aufgetrennten Features
df.head()

,episode_id,relevant,clear,optimistic,specific,cautious,assertive,valence,arousal,dominance,price_1d,price_30d,price_90d
0,CINF_2019_Q1,0.95,0.9,0.70,0.90,0.8,0.80,0.60,0.4,0.80,0.02,0.04,0.06
1,SMCI_2020_Q1,0.90,0.8,0.60,0.50,0.7,0.60,0.30,0.3,0.60,0.02,0.03,0.04
2,CI_2017_Q2,0.95,0.9,0.85,0.75,0.8,0.80,0.80,0.5,0.90,0.03,0.07,0.10
3,BLDR_2021_Q1,0.95,0.9,0.95,0.85,0.7,0.85,0.90,0.6,0.90,0.03,0.07,0.10
4,PSX_2018_Q2,0.95,0.9,0.90,0.85,0.7,0.80,0.85,0.6,0.85,0.02,0.05,0.08


In [11]:
# Verteilung der Feature-Werte prüfen
df.describe()

,relevant,clear,optimistic,specific,cautious,assertive,valence,arousal,dominance,price_1d,price_30d,price_90d
count,1358.000000,1358.000000,1358.000000,1358.000000,1358.000000,1358.000000,1358.000000,1358.000000,1358.000000,1358.000000,1358.000000,1358.000000
mean,0.928903,0.869219,0.712813,0.787813,0.745434,0.746797,0.569477,0.407143,0.766679,0.021863,0.044057,0.066024
std,0.030266,0.047280,0.135818,0.102388,0.073454,0.080086,0.191554,0.120856,0.072358,0.006189,0.021419,0.032323
min,0.500000,0.500000,0.200000,0.400000,0.200000,0.400000,0.100000,0.100000,0.500000,-0.030000,-0.080000,-0.120000
25%,0.900000,0.800000,0.600000,0.700000,0.700000,0.700000,0.500000,0.300000,0.700000,0.020000,0.040000,0.060000
50%,0.950000,0.900000,0.700000,0.800000,0.700000,0.800000,0.600000,0.400000,0.800000,0.020000,0.050000,0.070000
75%,0.950000,0.900000,0.800000,0.850000,0.800000,0.800000,0.700000,0.500000,0.800000,0.020000,0.050000,0.080000
max,1.000000,1.000000,1.000000,1.000000,0.900000,0.900000,1.000000,0.800000,1.000000,0.080000,0.120000,0.180000


In [12]:
# Qualitätskontrolle: Zeilen, in denen das Modell alle Dimensionen nur mit
# den Randwerten 0.0 oder 1.0 bewertet hat
feature_cols = [c for c in features.columns if c not in ['price_1d', 'price_30d', 'price_90d']]

mask = df[feature_cols].isin([0.0, 1.0]).all(axis=1)
count = mask.sum()

print(f"Rows with only 0.0 or 1.0 in feature columns: {count}")

Rows with only 0.0 or 1.0 in feature columns: 0


## 3 Distraction Bias testen

Verglichen werden die Ausgaben desselben Modells auf denselben Transkripten,
einmal anonymisiert (Versuch 2) und einmal im Original (Kontrollversuch). Die
Veränderung der Ausgabewerte entspricht demjenigen Anteil, der auf
unternehmensspezifisches Vorwissen zurückgeht.

Geprüft wird Hypothese 1: Reagiert die Feature-Extraktion schwächer auf die
Anonymisierung als die direkte Kursprognose, ist sie weniger anfällig für den
Distraction Bias.

In [21]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [14]:
# Anonymisierter Lauf aus Versuch 2 als Vergleichsgrundlage
df_v4 = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/ml_data_v4.csv')

In [18]:
# In beiden Läufen nur die 90-Tage-Prognose verwenden
df_v4 = df_v4.drop(columns=["price_1d", "price_30d", "move_1d", "move_30d", "move_next_qtr"])

In [13]:
df = df.drop(columns=["price_1d", "price_30d"])  # ungenutzt

In [22]:
# Paarweise Zuordnung beider Läufe über die episode_id
merged = df.merge(df_v4, on="episode_id", suffixes=("_orig", "_anon"))

In [24]:
# Stichprobengrösse und Spalten der zusammengeführten Tabelle prüfen
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   episode_id       1358 non-null   object 
 1   relevant_orig    1358 non-null   float64
 2   clear_orig       1358 non-null   float64
 3   optimistic_orig  1358 non-null   float64
 4   specific_orig    1358 non-null   float64
 5   cautious_orig    1358 non-null   float64
 6   assertive_orig   1358 non-null   float64
 7   valence_orig     1358 non-null   float64
 8   arousal_orig     1358 non-null   float64
 9   dominance_orig   1358 non-null   float64
 10  price_90d_orig   1358 non-null   float64
 11  relevant_anon    1358 non-null   float64
 12  clear_anon       1358 non-null   float64
 13  optimistic_anon  1358 non-null   float64
 14  specific_anon    1358 non-null   float64
 15  cautious_anon    1358 non-null   float64
 16  assertive_anon   1358 non-null   float64
 17  valence_anon  

In [46]:
sentiment_features = ["relevant", "clear", "optimistic", "specific",
                      "cautious", "assertive", "valence", "arousal", "dominance"]

# Standardisierte absolute Veränderung je Sentiment-Dimension. Die
# Standardisierung ist notwendig, da die Dimensionen unterschiedlich stark
# streuen und absolute Differenzen sonst nicht vergleichbar wären.
sentiment_changes = pd.DataFrame({
    feat: (merged[f"{feat}_orig"] - merged[f"{feat}_anon"]).abs() / merged[f"{feat}_orig"].std()
    for feat in sentiment_features
})

# Eine Zahl pro Transkript: mittlere standardisierte Veränderung über alle 9 Dimensionen
sentiment_change_agg = sentiment_changes.mean(axis=1)

# Standardisierte absolute Veränderung der Direktprognose (eine Zahl pro Transkript)
price_change = (merged["price_90d_orig"] - merged["price_90d_anon"]).abs() / merged["price_90d_orig"].std()

### 3.1 Hypothesentests

In [63]:
# Gepaarter t-Test: reagiert die Direktprognose stärker auf Anonymisierung als
# die Sentiment-Extraktion? Die Paarung ist entscheidend, da beide Grössen aus
# demselben Transkript und demselben LLM-Aufruf stammen.
t_stat, p_val = stats.ttest_rel(price_change, sentiment_change_agg)

# Effektstärke des Unterschieds (Cohen's d_z für gepaarte Daten:
# Mittelwert der Differenzen geteilt durch deren Streuung)
diff = price_change - sentiment_change_agg
d_z = diff.mean() / diff.std()

print(f"n = {len(merged)}")
print(f"Mittlere Veränderung Sentiment:       {sentiment_change_agg.mean():>5.3f}")
print(f"Mittlere Veränderung Direktprognose:  {price_change.mean():.3f}")
print(f"Differenz (Preis − Sentiment):       {diff.mean():.3f}")
print(f"Effektstärke (Cohen's d_z):          {d_z:.3f}")
print(f"""Paired t-Test:                   t = {t_stat:.3f}
                                 p = {p_val:.4f}""")

print()
if p_val < 0.05 and diff.mean() > 0:
    print("-> Ergebnis STÜTZT die Hypothese: Direktprognose reagiert stärker auf Anonymisierung.")
elif p_val < 0.05 and diff.mean() < 0:
    print("-> Ergebnis WIDERSPRICHT der Hypothese: Sentiment-Extraktion reagiert stärker.")
else:
    print("-> Kein signifikanter Unterschied zwischen den beiden Reaktionsstärken.")

n = 1358
Mittlere Veränderung Sentiment:       0.154
Mittlere Veränderung Direktprognose:  0.137
Differenz (Preis − Sentiment):       -0.018
Effektstärke (Cohen's d_z):          -0.047
Paired t-Test:                   t = -1.727
                                 p = 0.0843

-> Kein signifikanter Unterschied zwischen den beiden Reaktionsstärken.


In [50]:
# Ergänzend je Dimension einzeln testen, um zu prüfen, ob einzelne Dimensionen
# stärker oder schwächer als die Direktprognose reagieren
rows = []
for feat in sentiment_features:
    feat_change = sentiment_changes[feat]
    t, p = stats.ttest_rel(price_change, feat_change)
    rows.append({
        "Feature": feat,
        "mean_change": feat_change.mean(),
        "t_stat": t,
        "p_value": p
    })

# Korrektur des Signifikanzniveaus für multiples Testen nach Benjamini-Hochberg
breakdown = pd.DataFrame(rows)
breakdown["p_corrected"] = multipletests(breakdown["p_value"], method="fdr_bh")[1]
breakdown["significant"] = breakdown["p_corrected"] < 0.05
breakdown.sort_values("p_value")

,Feature,mean_change,t_stat,p_value,p_corrected,significant
0,relevant,0.242084,-5.736704,1.188807e-08,1.069926e-07,True
2,optimistic,0.085935,4.959928,7.943162e-07,3.574423e-06,True
8,dominance,0.195906,-4.059473,5.199272e-05,1.559782e-04,True
6,valence,0.099758,3.580818,3.546113e-04,7.978753e-04,True
7,arousal,0.179439,-3.260472,1.139871e-03,2.051768e-03,True
3,specific,0.170092,-2.641546,8.347679e-03,1.252152e-02,True
1,clear,0.169767,-1.836299,6.653210e-02,8.554127e-02,False
5,assertive,0.118153,1.463506,1.435604e-01,1.615055e-01,False
4,cautious,0.128320,0.583766,5.594749e-01,5.594749e-01,False
